# MeetingIQ — Synthetic Data Loader for Google Colab
**Scenario:** Rapid7 GTM world (`tenant_id = rapid7-gtm`)

Populates a **SQLite** database whose table names and columns mirror the production Postgres DDL
(`load_postgres.py`) exactly, so you can swap the connector to Postgres with zero schema changes.

**Tables created:**
`account` · `contact` · `opportunity` · `meeting` · `transcript` · `action_item` · `email` · `document`

**Synthetic world:**
- 7 internal Rapid7 users (AE, SE, MDR Specialist, Sales Manager, etc.)
- 10 prospect/customer accounts across Healthcare, Fintech, Manufacturing, Banking…
- 54 buying-committee contacts (hero committees hand-authored for A1/A5/A8)
- 14 opportunities with realistic ACV and stage
- 68 meetings across the full sales arc (discovery → close/loss debrief)
- 53 transcripts with planted deal signals (budget freezes, competitor mentions, champion changes)
- 118 emails (recap, scheduling, intro, at-risk threads)
- 30 documents (proposals, battlecards, POC criteria)
- 79 action items

> **No API keys needed.** All data is deterministic (fixed `SEED = 20260608`).
> Transcript bodies and email bodies are realistic stubs — run Stage 2 (`expand_llm.py`) with
> `ANTHROPIC_API_KEY` to LLM-expand them into full prose.

## 1 · Install dependencies

In [ ]:
!pip install -q faker polars
print("Dependencies ready")

## 2 · Configuration

In [ ]:
import sqlite3, random, json
from datetime import datetime, date, timedelta
from faker import Faker

# ── reproducibility (matches config.py) ──────────────────────────────────────

SEED        = 20260608
TENANT_ID   = "rapid7-gtm"
DEMO_TODAY  = date(2026, 6, 8) - timedelta(days=date(2026, 6, 8).weekday())  # Monday 2026-06-02
HISTORY_DAYS  = 130
UPCOMING_DAYS = 14

DB_PATH = "meetingiq.db"   # change to "/content/drive/MyDrive/meetingiq.db" for Drive

random.seed(SEED)
Faker.seed(SEED)
fake = Faker()

conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = ON")
cur  = conn.cursor()
print(f"✅ Connected → {DB_PATH}  (DEMO_TODAY = {DEMO_TODAY})")

## 3 · Schema — mirrors production Postgres DDL exactly

In [ ]:
DDL = """
-- mirrors load_postgres.py DDL (table names = Salesforce object names)

CREATE TABLE IF NOT EXISTS account (
    id          TEXT PRIMARY KEY,
    external_id TEXT,
    tenant_id   TEXT NOT NULL,
    name        TEXT NOT NULL,
    vertical    TEXT,
    size        INTEGER,
    colour      TEXT CHECK(colour IN ('green','yellow','red','grey','blue')),
    state       TEXT
);

CREATE TABLE IF NOT EXISTS contact (
    id              TEXT PRIMARY KEY,
    external_id     TEXT,
    tenant_id       TEXT NOT NULL,
    account_id      TEXT NOT NULL REFERENCES account(id),
    name            TEXT NOT NULL,
    title           TEXT,
    band            TEXT,
    committee_role  TEXT CHECK(committee_role IN
                        ('economic_buyer','champion','technical_evaluator','blocker','influencer')),
    decision_power  TEXT CHECK(decision_power IN ('low','medium','high')),
    email           TEXT
);

CREATE TABLE IF NOT EXISTS opportunity (
    id          TEXT PRIMARY KEY,
    external_id TEXT,
    tenant_id   TEXT NOT NULL,
    account_id  TEXT NOT NULL REFERENCES account(id),
    name        TEXT NOT NULL,
    stage       TEXT,
    acv         INTEGER,
    is_closed   INTEGER DEFAULT 0,   -- SQLite bool
    is_won      INTEGER DEFAULT 0
);

CREATE TABLE IF NOT EXISTS meeting (
    id              TEXT PRIMARY KEY,
    external_id     TEXT,
    tenant_id       TEXT NOT NULL,
    account_id      TEXT NOT NULL REFERENCES account(id),
    opportunity_id  TEXT REFERENCES opportunity(id),
    title           TEXT NOT NULL,
    meeting_type    TEXT,
    channel         TEXT,
    recorded        INTEGER DEFAULT 0,
    start_ts        TEXT NOT NULL,
    duration_min    INTEGER DEFAULT 30
);

CREATE TABLE IF NOT EXISTS transcript (
    id          TEXT PRIMARY KEY,
    tenant_id   TEXT NOT NULL,
    meeting_id  TEXT NOT NULL REFERENCES meeting(id),
    origin      TEXT CHECK(origin IN ('text','whisper')),
    text        TEXT NOT NULL DEFAULT ''
);

CREATE TABLE IF NOT EXISTS action_item (
    id          TEXT PRIMARY KEY,
    tenant_id   TEXT NOT NULL,
    account_id  TEXT REFERENCES account(id),
    meeting_id  TEXT REFERENCES meeting(id),
    title       TEXT NOT NULL,
    owner       TEXT,
    status      TEXT DEFAULT 'open' CHECK(status IN ('open','in_progress','done','overdue')),
    due         TEXT
);

CREATE TABLE IF NOT EXISTS email (
    id          TEXT PRIMARY KEY,
    tenant_id   TEXT NOT NULL,
    thread_id   TEXT,
    account_id  TEXT REFERENCES account(id),
    subject     TEXT,
    from_addr   TEXT,
    body        TEXT DEFAULT ''
);

CREATE TABLE IF NOT EXISTS document (
    id          TEXT PRIMARY KEY,
    tenant_id   TEXT NOT NULL,
    account_id  TEXT REFERENCES account(id),
    doc_type    TEXT CHECK(doc_type IN
                    ('proposal','battlecard','poc_criteria',
                     'security_questionnaire','order_form','report')),
    title       TEXT NOT NULL,
    body        TEXT DEFAULT ''
);
"""

conn.executescript(DDL)
conn.commit()
print("✅ Schema created — 8 tables")

## 4 · Master data definitions (from `accounts.py`)

In [ ]:
# ── Internal Rapid7 team (from accounts.py USERS) ────────────────────────────
USERS = [
    {"id": "U1", "name": "Priya Menon",  "role": "Enterprise AE",   "email": "priya.menon@rapid7.example"},
    {"id": "U2", "name": "Dev Sharma",   "role": "Sales Engineer",   "email": "dev.sharma@rapid7.example"},
    {"id": "U3", "name": "Lena Ortiz",   "role": "MDR Specialist",   "email": "lena.ortiz@rapid7.example"},
    {"id": "U4", "name": "Marcus Hale",  "role": "Sales Manager",    "email": "marcus.hale@rapid7.example"},
    {"id": "U5", "name": "Aisha Khan",   "role": "Pre-sales Lead",   "email": "aisha.khan@rapid7.example"},
    {"id": "U6", "name": "Tom Becker",   "role": "SDR",              "email": "tom.becker@rapid7.example"},
    {"id": "U7", "name": "Ravi Nair",    "role": "Deal Desk",        "email": "ravi.nair@rapid7.example"},
]
AE_ID = "U1"   # Priya Menon — primary owner

# ── 10 Accounts (from accounts.py ACCOUNTS) ───────────────────────────────────
ACCOUNTS = [
    {"id":"A1",  "name":"Meridian Health Systems",  "vertical":"Healthcare",     "size":2000,  "colour":"green",  "state":"open_advancing",
     "competitor":"Tenable",              "acv":280000, "hero":True,  "products":["Exposure Command Ultimate","MDR"]},
    {"id":"A2",  "name":"Northwind Logistics",      "vertical":"Transportation", "size":8000,  "colour":"green",  "state":"open_advancing",
     "competitor":"Microsoft Sentinel",   "acv":420000, "hero":False, "products":["InsightIDR","InsightCloudSec"]},
    {"id":"A3",  "name":"Cobalt Pay",               "vertical":"Fintech",        "size":1200,  "colour":"yellow", "state":"open_new",
     "competitor":"Wiz",                  "acv":190000, "hero":False, "products":["Exposure Command Ultimate"]},
    {"id":"A4",  "name":"Larkspur Retail Group",    "vertical":"Retail",         "size":12000, "colour":"yellow", "state":"open_new",
     "competitor":"Qualys",               "acv":260000, "hero":False, "products":["Surface Command","InsightVM"]},
    {"id":"A5",  "name":"Vantage Semiconductors",   "vertical":"Manufacturing",  "size":15000, "colour":"red",    "state":"at_risk",
     "competitor":"Tenable",              "acv":510000, "hero":True,  "products":["Exposure Command Ultimate","InsightAppSec"]},
    {"id":"A6",  "name":"Aspen Digital",            "vertical":"Tech SaaS",      "size":900,   "colour":"red",    "state":"at_risk",
     "competitor":"Arctic Wolf",          "acv":150000, "hero":False, "products":["MDR"]},
    {"id":"A7",  "name":"Granite State Bank",       "vertical":"Banking",        "size":5000,  "colour":"green",  "state":"closed_won",
     "competitor":"Qualys",               "acv":360000, "hero":False, "products":["InsightVM","Exposure Command Essentials"]},
    {"id":"A8",  "name":"Helios Energy",            "vertical":"Energy",         "size":7000,  "colour":"grey",   "state":"closed_lost",
     "competitor":"CrowdStrike",          "acv":340000, "hero":True,  "products":["Exposure Command Ultimate"]},
    {"id":"A9",  "name":"Brightwave Media",         "vertical":"Media",          "size":1500,  "colour":"green",  "state":"expansion",
     "competitor":None,                   "acv":120000, "hero":False, "products":["InsightCloudSec"]},
    {"id":"A10", "name":"Pinnacle Insurance",       "vertical":"Insurance",      "size":10000, "colour":"blue",   "state":"steady",
     "competitor":"Recorded Future",      "acv":95000,  "hero":False, "products":["Intelligence Hub","InsightIDR"]},
]

# ── Hero committees (A1/A5/A8) — hand-authored from accounts.py ───────────────
HERO_COMMITTEES = {
    "A1": [
        {"name":"Dana Reyes",   "title":"CISO",                    "band":"C-level",  "role":"economic_buyer",     "power":"high"},
        {"name":"Sam Okafor",   "title":"Director, Security Ops",  "band":"Director", "role":"champion",           "power":"high"},
        {"name":"Joel Tan",     "title":"SOC Manager",             "band":"Manager",  "role":"technical_evaluator","power":"medium"},
        {"name":"Grace Liu",    "title":"Procurement Manager",     "band":"Manager",  "role":"blocker",            "power":"medium"},
        {"name":"Anil Patel",   "title":"Director, Compliance",    "band":"Director", "role":"influencer",         "power":"medium"},
    ],
    "A5": [
        {"name":"Erik Vance",   "title":"CISO",                    "band":"C-level",  "role":"economic_buyer",     "power":"high"},
        {"name":"Nadia Roy",    "title":"Vulnerability Mgmt Lead", "band":"Manager",  "role":"champion",           "power":"medium"},
        {"name":"Tomas Ruiz",   "title":"Director, Procurement",   "band":"Director", "role":"blocker",            "power":"high"},
        {"name":"Karen Webb",   "title":"Finance Business Partner","band":"Director", "role":"influencer",         "power":"high"},
    ],
    "A8": [
        {"name":"Marco Bianchi","title":"CISO",                    "band":"C-level",  "role":"economic_buyer",     "power":"high"},
        {"name":"Lauren Cho",   "title":"SecOps Lead",             "band":"Manager",  "role":"champion",           "power":"medium"},
        {"name":"Greg Mason",   "title":"SOC Analyst",             "band":"IC",       "role":"technical_evaluator","power":"low"},
    ],
}

ROLE_TEMPLATE = [
    ("economic_buyer",     "high",   "C-level",  "CISO"),
    ("champion",           "high",   "Director", "Director, Security Ops"),
    ("technical_evaluator","medium", "Manager",  "SOC Manager"),
    ("technical_evaluator","medium", "Manager",  "Security Analyst"),
    ("blocker",            "medium", "Manager",  "Procurement Manager"),
    ("influencer",         "medium", "Director", "Director, Compliance"),
]

# ── Meeting arc per deal state (from skeleton.py STATE_ARC) ───────────────────
STATE_ARC = {
    "open_new":       ["sdr_qualification","discovery","technical_demo","poc_kickoff"],
    "open_advancing": ["discovery","technical_demo","poc_kickoff","poc_checkin",
                       "poc_results_business_case","pricing_packaging"],
    "at_risk":        ["discovery","technical_demo","poc_kickoff","poc_checkin",
                       "poc_results_business_case","pricing_packaging","negotiation"],
    "closed_won":     ["discovery","technical_demo","poc_kickoff","poc_results_business_case",
                       "pricing_packaging","negotiation","close_or_loss_debrief"],
    "closed_lost":    ["discovery","technical_demo","poc_kickoff","poc_results_business_case",
                       "pricing_packaging","close_or_loss_debrief"],
    "expansion":      ["discovery","technical_demo","poc_results_business_case"],
    "steady":         ["recurring_sync","recurring_sync","recurring_sync"],
}
STAGE_BY_STATE = {
    "open_new":"Discovery", "open_advancing":"Technical Validation",
    "at_risk":"Negotiation (stalled)", "closed_won":"Closed Won",
    "closed_lost":"Closed Lost", "expansion":"Expansion", "steady":"Customer Success",
}
RECORDED_TYPES = {"discovery","technical_demo","poc_kickoff","poc_results_business_case",
                  "pricing_packaging","negotiation","recurring_sync"}

# ── Transcript stubs: [persona:role] format required by tts.py for voice assignment ──
# Format: [persona:role] Name: line  — Stage 2 LLM expands these; Stage 3 TTS splits on them.
# {competitor} is replaced at runtime per account by _stub().
TRANSCRIPT_STUBS = {
    "sdr_qualification":
        "[persona:salesperson] Tom Becker: Hi, I'm reaching out because we've been helping "
        "companies like yours reduce their exposure surface. Do you have 30 minutes for a discovery call?\n"
        "[persona:contact] Prospect: Sure, we've been struggling with prioritising vulnerabilities "
        "across cloud and on-prem environments.\n"
        "[persona:salesperson] Tom Becker: Exactly the pain we solve. Let me set up a call with our AE.",
    "discovery":
        "[persona:salesperson] Priya Menon: Thanks for joining — I'd love to understand your current "
        "vulnerability management workflow. What's your biggest pain point today?\n"
        "[persona:contact] Champion: We're running {competitor} but coverage gaps keep appearing in cloud assets.\n"
        "[persona:presales] Dev Sharma: That's exactly what Exposure Command addresses — unified visibility "
        "across hybrid environments with no agent sprawl.\n"
        "[persona:contact] Champion: That would be a big improvement. Can we see a demo?",
    "technical_demo":
        "[persona:presales] Dev Sharma: Let me walk you through the asset inventory view — "
        "everything auto-discovered, no agents required.\n"
        "[persona:contact] Technical Evaluator: Can this integrate with our existing SIEM via API?\n"
        "[persona:presales] Dev Sharma: Yes — we have native connectors for Splunk, Sentinel, "
        "and a REST API for custom integrations.\n"
        "[persona:contact] Champion: The risk prioritisation view is impressive. How does scoring work?\n"
        "[persona:presales] Dev Sharma: We combine CVSS, asset criticality, and real threat intel "
        "to surface what actually needs patching first.",
    "poc_kickoff":
        "[persona:salesperson] Priya Menon: Let's lock in the POC success criteria. "
        "We're targeting a 40% reduction in critical vulns over 30 days.\n"
        "[persona:contact] Champion: Agreed. We'll share the asset inventory list by end of week.\n"
        "[persona:presales] Dev Sharma: I'll set up the tenant and we can run the first scan on Friday.\n"
        "[persona:contact] Champion: Works for us. Who do we contact if we hit issues?",
    "poc_checkin":
        "[persona:presales] Dev Sharma: Early POC results look strong — 34% reduction in critical "
        "vulns in week one.\n"
        "[persona:contact] Champion: The team is impressed. Can we extend coverage to the OT segment?\n"
        "[persona:contact] Blocker: We need to confirm this fits the procurement timeline "
        "before we can go any further.\n"
        "[persona:salesperson] Priya Menon: Understood — let's get procurement aligned this week "
        "so it doesn't block the close.",
    "poc_results_business_case":
        "[persona:presales] Dev Sharma: Final POC results: 47% reduction in critical exposure "
        "across 12,000 assets. Here's the full business case breakdown.\n"
        "[persona:contact] Economic Buyer: These numbers are compelling. What does the full "
        "deployment timeline look like?\n"
        "[persona:salesperson] Priya Menon: We can be fully deployed within 60 days with a "
        "dedicated onboarding team.\n"
        "[persona:contact] Economic Buyer: And what's the impact on our current {competi tor} licence?",
    "pricing_packaging":
        "[persona:salesperson] Priya Menon: We have three tiers — here's how each maps to "
        "your environment and headcount.\n"
        "[persona:contact] Economic Buyer: Budget cycle closes Q3, so timing matters. "
        "How does this compare to {competitor} on total cost?\n"
        "[persona:salesperson] Priya Menon: We're competitive on TCO when you factor in "
        "the MDR coverage and reduced FTE overhead included.\n"
        "[persona:contact] Champion: The Ultimate tier looks like the right fit for our scale.",
    "negotiation":
        "[persona:salesperson] Priya Menon: Legal has reviewed the redlines — "
        "here's our final position on the master terms.\n"
        "[persona:contact] Blocker: Procurement needs net-60 payment terms as a hard requirement.\n"
        "[persona:salesperson] Priya Menon: I can offer 10% off the first year if we sign "
        "by month-end with net-45.\n"
        "[persona:contact] Champion: That works on our end — let me get sign-off from finance today.",
    "close_or_loss_debrief":
        "[persona:salesperson] Priya Menon: Let's walk through the deal outcome and "
        "capture the key learnings for the forecast.\n"
        "[persona:salesperson] Marcus Hale: What were the top two factors that determined the result?\n"
        "[persona:salesperson] Priya Menon: Champion strength and procurement timing. "
        "We'll adjust the playbook for similar deals next quarter.",
    "recurring_sync":
        "[persona:salesperson] Priya Menon: Quick check on open action items from last month.\n"
        "[persona:contact] Champion: The integration ticket is in progress — should close this sprint.\n"
        "[persona:salesperson] Priya Menon: Good. Any expansion areas you'd like us to scope "
        "for the renewal conversation?\n"
        "[persona:contact] Champion: We're looking at adding InsightAppSec for the dev teams.",
    "internal_sync":
        "[persona:salesperson] Priya Menon: Let's align on deal risk and next steps "
        "before the exec review tomorrow.\n"
        "[persona:salesperson] Marcus Hale: What's blocking the economic buyer from committing?\n"
        "[persona:presales] Dev Sharma: The blocker is procurement timeline — "
        "they need a net-60 clause that we haven't agreed yet.",
}

print(f"✅ Master data: {len(USERS)} users · {len(ACCOUNTS)} accounts")
print(f"   TRANSCRIPT_STUBS: {len(TRANSCRIPT_STUBS)} meeting types loaded with [persona:role] format")


## 5 · Helper functions

In [ ]:
from typing import Optional

def _meeting_dates(n: int, state: str) -> list:
    """Return n ISO datetime strings spread realistically across the sales arc."""
    today = datetime(DEMO_TODAY.year, DEMO_TODAY.month, DEMO_TODAY.day, 10, 0)
    if state in ("closed_won", "closed_lost"):
        start = today - timedelta(days=HISTORY_DAYS)
        end   = today - timedelta(days=18)
        future = 0
    elif state == "steady":
        base = today - timedelta(days=40)
        return [(base + timedelta(days=30*i, hours=random.randint(-2,5))).isoformat() for i in range(n)]
    else:
        start  = today - timedelta(days=HISTORY_DAYS)
        end    = today + timedelta(days=UPCOMING_DAYS)
        future = 1 if state == "open_new" else 2

    span   = (end - start).days
    past_n = n - future
    dates  = []
    for i in range(past_n):
        d = start + timedelta(days=int(span*(i/max(past_n,1))*0.7) + random.randint(0,4))
        dates.append(d.replace(hour=random.choice([9,11,14,15]), minute=0))
    for j in range(future):
        d = today + timedelta(days=2 + j*5 + random.randint(0,2))
        dates.append(d.replace(hour=random.choice([10,13]), minute=0))
    return [d.isoformat() for d in sorted(dates)]

def _committee(acc: dict) -> list:
    """Return hand-authored hero committee or deterministic template."""
    if acc["id"] in HERO_COMMITTEES:
        return HERO_COMMITTEES[acc["id"]]
    return [{"name": fake.name(), "title": title, "band": band, "role": role, "power": power}
            for role, power, band, title in ROLE_TEMPLATE]

# FIX Bug 2: use Optional[str] instead of str | None for Python 3.9 compatibility
def _stub(mtype: str, competitor: Optional[str]) -> str:
    """Return the transcript stub for a meeting type, interpolating the competitor name."""
    tmpl = TRANSCRIPT_STUBS.get(
        mtype,
        "[persona:salesperson] Priya Menon: [stub] Meeting transcript.\n"
        "[persona:contact] Prospect: [stub] Response."
    )
    return tmpl.replace("{competitor}", competitor or "incumbent vendor")

print("✅ Helpers defined")


## 6 · Insert internal users

In [ ]:
# Users live in the account table with a special sentinel account_id = NULL;
# we store them separately here for reference only (they are owners/attendees, not accounts).
# In the Postgres DDL there is no separate `user` table — users are referenced by email string.
# We'll print them for reference:
print("Internal Rapid7 team:")
for u in USERS:
    print(f"  [{u['id']}] {u['name']:20s}  {u['role']}")

## 7 · Generate & insert all entities

In [ ]:
audio_budget  = 12
audio_count   = 0
# hero/active accounts get audio allocation first
priority_ids  = {"A1","A2","A5","A6","A8","A9"}

all_accounts     = []
all_contacts     = []
all_opps         = []
all_meetings     = []
all_transcripts  = []
all_action_items = []
all_emails       = []
all_documents    = []

# FIX Bug 3: track contacts per account for safe lookup (avoid fragile negative index)
contacts_by_account = {}

for acc in ACCOUNTS:
    aid   = acc["id"]
    state = acc["state"]

    # ── account ───────────────────────────────────────────────────────────────
    all_accounts.append((aid, None, TENANT_ID, acc["name"], acc["vertical"], acc["size"],
                          acc["colour"], state))

    # ── contacts (buying committee) ────────────────────────────────────────────
    committee = _committee(acc)
    cids = []
    contacts_by_account[aid] = []
    for k, c in enumerate(committee):
        cid = f"{aid}-C{k+1}"
        cids.append(cid)
        email = f"{c['name'].lower().replace(' ','.').replace(chr(39),'')}@{acc['name'].lower().replace(' ','')}.example"
        row = (cid, None, TENANT_ID, aid, c["name"], c["title"],
               c["band"], c["role"], c["power"], email)
        all_contacts.append(row)
        contacts_by_account[aid].append(row)

    # ── primary opportunity ────────────────────────────────────────────────────
    oid       = f"{aid}-OPP1"
    is_closed = 1 if state.startswith("closed") else 0
    is_won    = 1 if state == "closed_won"       else 0
    all_opps.append((oid, None, TENANT_ID, aid,
                     f"{acc['name']} — {acc['products'][0]}",
                     STAGE_BY_STATE[state], acc["acv"], is_closed, is_won))

    # ── meetings along the arc ─────────────────────────────────────────────────
    arc   = STATE_ARC[state]
    dates = _meeting_dates(len(arc), state)

    for k, (mtype, start_ts) in enumerate(zip(arc, dates)):
        mid      = f"{aid}-M{k+1}"
        recorded = 1 if mtype in RECORDED_TYPES else 0
        internal = 1 if mtype == "close_or_loss_debrief" else 0
        channel  = "internal" if internal else random.choice(["zoom","gmeet"])
        dur      = random.choice([30, 45, 60])
        audio    = (recorded and audio_count < audio_budget and aid in priority_ids)
        if audio:
            audio_count += 1

        all_meetings.append((mid, None, TENANT_ID, aid, oid,
                              f"{acc['name']}: {mtype.replace('_',' ').title()}",
                              mtype, channel, recorded, start_ts, dur))

        # transcript stub (origin = whisper for audio meetings, text otherwise)
        origin = "whisper" if audio else "text"
        stub   = _stub(mtype, acc["competitor"])
        all_transcripts.append((f"TR-{mid}", TENANT_ID, mid, origin, stub))

        # action items (1-2 per meeting)
        for ai in range(random.randint(1, 2)):
            due = (datetime.fromisoformat(start_ts) + timedelta(days=3)).date().isoformat()
            all_action_items.append((f"{mid}-T{ai+1}", TENANT_ID, aid, mid,
                                     f"Follow-up after {mtype.replace('_',' ')}", AE_ID, "open", due))

        # FIX Bug 3: look up contact email by account id — no fragile negative index
        contact_email = contacts_by_account[aid][0][9]  # first committee member email
        all_emails.append((f"{mid}-E-recap", TENANT_ID, f"{aid}-thread", aid,
                           f"Recap: {mtype.replace('_',' ')}", "priya.menon@rapid7.example",
                           f"[stub] Hi, thanks for today's {mtype.replace('_',' ')} — here's a quick recap."))
        all_emails.append((f"{mid}-E-sched", TENANT_ID, f"{aid}-sched", aid,
                           f"Scheduling: next {mtype.replace('_',' ')}", "priya.menon@rapid7.example",
                           f"[stub] Would any of these times work for the follow-up?"))

    # ── internal war-room syncs ────────────────────────────────────────────────
    n_sync = (2 if state in ("open_advancing","at_risk","closed_won","closed_lost")
              else 1 if state in ("open_new","expansion") else 0)
    for sci in range(n_sync):
        sdt = (datetime.fromisoformat(dates[min(sci+1, len(dates)-1)])
               - timedelta(days=1)).isoformat()
        all_meetings.append((f"{aid}-IS{sci+1}", None, TENANT_ID, aid, oid,
                              f"{acc['name']}: internal deal sync",
                              "internal_sync","internal", 0, sdt, 30))
        all_transcripts.append((f"TR-{aid}-IS{sci+1}", TENANT_ID, f"{aid}-IS{sci+1}",
                                 "text", TRANSCRIPT_STUBS["internal_sync"]))

    # ── intro + at-risk emails ─────────────────────────────────────────────────
    all_emails.append((f"{aid}-E-intro", TENANT_ID, f"{aid}-intro", aid,
                       f"Intro: Rapid7 × {acc['name']}","tom.becker@rapid7.example",
                       "[stub] Hi, I'd love to connect and share how Rapid7 can help."))
    if state == "at_risk":
        all_emails.append((f"{aid}-E-quiet", TENANT_ID, f"{aid}-quiet", aid,
                           "Checking in","priya.menon@rapid7.example",
                           "[stub] Just wanted to see if there's anything we can do to move things forward."))

    # ── documents ─────────────────────────────────────────────────────────────
    for dt in ["proposal","battlecard","poc_criteria"]:
        all_documents.append((f"{aid}-DOC-{dt}", TENANT_ID, aid, dt,
                               f"{acc['name']} {dt.replace('_',' ')}", ""))

    # ── second expansion opportunity for selected accounts ─────────────────────
    if aid in ("A1","A2","A7","A9"):
        all_opps.append((f"{aid}-OPP2", None, TENANT_ID, aid,
                         f"{acc['name']} — Expansion ({acc['products'][-1]})",
                         "Expansion", int(acc["acv"]*0.4), 0, 0))

# FIX Bug 4: assert audio count matches expected budget so failures are loud
assert audio_count == 12, f"Expected 12 audio meetings, got {audio_count} — check priority_ids or RECORDED_TYPES"

print(f"Generated:")
print(f"  accounts        {len(all_accounts)}")
print(f"  contacts        {len(all_contacts)}")
print(f"  opportunities   {len(all_opps)}")
print(f"  meetings        {len(all_meetings)}")
print(f"  transcripts     {len(all_transcripts)}")
print(f"  action_items    {len(all_action_items)}")
print(f"  emails          {len(all_emails)}")
print(f"  documents       {len(all_documents)}")
print(f"  audio meetings  {audio_count}  ✅")


## 8 · Bulk-insert into SQLite

In [ ]:
cur.executemany(
    "INSERT OR IGNORE INTO account VALUES (?,?,?,?,?,?,?,?)", all_accounts)

cur.executemany(
    "INSERT OR IGNORE INTO contact VALUES (?,?,?,?,?,?,?,?,?,?)", all_contacts)

cur.executemany(
    "INSERT OR IGNORE INTO opportunity VALUES (?,?,?,?,?,?,?,?,?)", all_opps)

cur.executemany(
    "INSERT OR IGNORE INTO meeting VALUES (?,?,?,?,?,?,?,?,?,?,?)", all_meetings)

cur.executemany(
    "INSERT OR IGNORE INTO transcript VALUES (?,?,?,?,?)", all_transcripts)

cur.executemany(
    "INSERT OR IGNORE INTO action_item VALUES (?,?,?,?,?,?,?,?)", all_action_items)

cur.executemany(
    "INSERT OR IGNORE INTO email VALUES (?,?,?,?,?,?,?)", all_emails)

cur.executemany(
    "INSERT OR IGNORE INTO document VALUES (?,?,?,?,?,?)", all_documents)

conn.commit()
print("✅ All rows committed.")

## 9 · Verification

In [ ]:
TABLES = ["account","contact","opportunity","meeting","transcript","action_item","email","document"]
print("=" * 40)
print(f"  {'Table':<20} Rows")
print("=" * 40)
for t in TABLES:
    n = cur.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"  {t:<20} {n}")
print("=" * 40)

## 10 · Sample queries

In [ ]:
# ── Q1: Deal health dashboard ─────────────────────────────────────────────────
print("\n📊 Deal health (colour/state breakdown)\n")
rows = cur.execute("""
    SELECT colour, state, COUNT(*) AS deals, SUM(acv) AS total_acv
    FROM account a
    JOIN opportunity o ON o.account_id = a.id
    WHERE o.id LIKE '%-OPP1'
    GROUP BY colour, state
    ORDER BY total_acv DESC
""").fetchall()
for colour, state, n, acv in rows:
    print(f"  {colour:<8} {state:<22} deals={n}  ACV=${acv:,}")

In [ ]:
# ── Q2: Open action items per owner ──────────────────────────────────────────
print("\n📋 Open action items per owner\n")
rows = cur.execute("""
    SELECT owner, COUNT(*) AS n, MIN(due) AS earliest_due
    FROM action_item
    WHERE status = 'open'
    GROUP BY owner
    ORDER BY n DESC
""").fetchall()
for owner, n, due in rows:
    print(f"  {owner:<35} items={n}  earliest_due={due}")

In [ ]:
# ── Q3: Meetings with whisper transcripts (the 12 audio calls) ───────────────
print("\n🎙  Whisper-transcribed calls\n")
rows = cur.execute("""
    SELECT m.title, m.start_ts, t.origin
    FROM transcript t
    JOIN meeting m ON m.id = t.meeting_id
    WHERE t.origin = 'whisper'
    ORDER BY m.start_ts
""").fetchall()
print(f"  Total: {len(rows)}")
for title, start, origin in rows[:6]:
    print(f"  [{origin}] {start[:10]}  {title}")
if len(rows) > 6:
    print(f"  … and {len(rows)-6} more")

In [ ]:
# ── Q4: Buying committee coverage for hero accounts ───────────────────────────
print("\n🧑‍💼 Buying committee — hero accounts (A1, A5, A8)\n")
rows = cur.execute("""
    SELECT a.name AS account, c.name, c.title, c.committee_role, c.decision_power
    FROM contact c
    JOIN account a ON a.id = c.account_id
    WHERE a.id IN ('A1','A5','A8')
    ORDER BY a.id, c.committee_role
""").fetchall()
last_acc = None
for acc, name, title, role, power in rows:
    if acc != last_acc:
        print(f"\n  {acc}")
        last_acc = acc
    print(f"    {role:<22} {name:<20} ({title}) — power:{power}")

In [ ]:
# ── Q5: Pipeline summary ─────────────────────────────────────────────────────
print("\n💰 Pipeline summary\n")
row = cur.execute("""
    SELECT
        SUM(CASE WHEN is_closed=0 THEN acv ELSE 0 END)             AS open_pipeline,
        SUM(CASE WHEN is_closed=1 AND is_won=1 THEN acv ELSE 0 END) AS closed_won,
        SUM(CASE WHEN is_closed=1 AND is_won=0 THEN acv ELSE 0 END) AS closed_lost
    FROM opportunity
""").fetchone()
print(f"  Open pipeline : ${row[0]:>10,}")
print(f"  Closed won    : ${row[1]:>10,}")
print(f"  Closed lost   : ${row[2]:>10,}")

## 11 · Done

`meetingiq.db` is ready.

```python
# Download from Colab:
from google.colab import files
files.download('meetingiq.db')
```

**Next steps:**
- Run `expand_llm.py` (Stage 2) with `ANTHROPIC_API_KEY` to replace `[stub]` bodies with real LLM-generated transcripts, emails, and documents.
- Run `load_postgres.py` against `docker compose up -d postgres` to land the same data in Postgres (the DDL is identical).
- The `ground_truth_ledger.json` is not stored in SQLite — it lives as a JSON sidecar at `out/ground_truth_ledger.json`.

In [ ]:
conn.close()
print("✅ Connection closed. meetingiq.db is ready!")

# Uncomment to download:
# from google.colab import files
# files.download('meetingiq.db')